# 02 — Explore Training Data

[Notebook guide](README.md) · [Project overview](../README.md) · [English paper](../docs/en/paper.pdf)

Inspect the dataset used in the recorded training run.

| Item | Details |
| --- | --- |
| Inputs | `final_train_dataset.json` in the working directory. |
| Outputs | Saved descriptive statistics and frequency tables; no training-data rewrite. |
| Requirements | Python with datasets, pandas, and scikit-learn; CPU is sufficient. |
| Configuration overview | Input filename in the loading cell; script detection on the instruction field; unigram/bigram frequency analysis with 20 features. |

**Recorded experiment:** run code cells in their existing order. This presentation edit preserves executable code, parameters, prompts, data, and experimental outputs. Configuration is summarized here without moving assignments. Installation logs are omitted; execution counts belong to the original run. Serbian prompts, reference text, docstrings, and printed outputs remain verbatim.


## 1. Load the combined dataset

In [6]:
from datasets import load_dataset

train_dataset = load_dataset("json", data_files="final_train_dataset.json", split="train")

Generating train split: 3152 examples [00:00, 144140.76 examples/s]


## 2. Inspect missing values

In [7]:
import pandas as pd
import re

# Convert to a pandas DataFrame for analysis
df = train_dataset.to_pandas()

# 1. Inspect missing values
print("--- Prazne vrednosti po kolonama ---")
print(df.isnull().sum())



--- Prazne vrednosti po kolonama ---
instruction    0
input          0
output         0
dtype: int64


## 3. Inspect text lengths

In [8]:
# 2. Text-length statistics (characters)
df['total_len'] = (df['instruction'].fillna('') + df['input'].fillna('') + df['output'].fillna('')).str.len()
print("\n--- Deskriptivna statistika dužine teksta ---")
print(df['total_len'].describe())




--- Deskriptivna statistika dužine teksta ---
count    3152.000000
mean      767.123096
std       557.815195
min        31.000000
25%       294.000000
50%       670.500000
75%      1055.750000
max      3048.000000
Name: total_len, dtype: float64


## 4. Inspect script balance

In [9]:
# 3. Script balance (detect Cyrillic in instructions)
def is_cyrillic(text):
    return bool(re.search(r'[\u0400-\u04FF]', str(text)))

cyr_count = df['instruction'].apply(is_cyrillic).sum()
lat_count = len(df) - cyr_count

print("\n--- Balans pisma ---")
print(f"Ćirilica: {cyr_count} ({cyr_count/len(df)*100:.1f}%)")
print(f"Latinica: {lat_count} ({lat_count/len(df)*100:.1f}%)")


--- Balans pisma ---
Ćirilica: 1649 (52.3%)
Latinica: 1503 (47.7%)


## 5. Inspect frequent terms

In [10]:
# 4. Most frequent terms and phrases
from sklearn.feature_extraction.text import CountVectorizer

df = train_dataset.to_pandas()

# Combine instruction and input for text analysis
text_data = df['instruction'].fillna('') + " " + df['input'].fillna('')

# Serbian stop words excluded from the frequency analysis
sr_stopwords = [
    # Latin script
    "je", "da", "se", "i", "u", "na", "za", "koji", "koja", "koje", "sa", "sam",
    "su", "ne", "iz", "od", "a", "ali", "ili", "kako", "šta", "sto", "kao", "to",
    "bi", "o", "pa", "do", "sve", "zbog", "po", "nakon", "tako", "tada", "samo", "ima",
    # Cyrillic script
    "је", "да", "се", "и", "у", "на", "за", "који", "која", "које", "са", "сам",
    "су", "не", "из", "од", "а", "али", "или", "како", "шта", "сто", "као", "то",
    "би", "о", "па", "до", "све", "због", "по", "након", "тако", "тада", "само", "има"
]

# N-gram analysis: frequent individual words and two-word phrases
vectorizer = CountVectorizer(ngram_range=(1, 2), stop_words=sr_stopwords, max_features=20)
X = vectorizer.fit_transform(text_data)

# Tabulate the most frequent terms
top_phrases = pd.DataFrame({
    'fraza/pojam': vectorizer.get_feature_names_out(),
    'broj_pojavljivanja': X.sum(axis=0).A1
}).sort_values(by='broj_pojavljivanja', ascending=False)

print("--- Top 20 najzastupljenijih pojmova u dataset-u ---")
print(top_phrases.to_string(index=False))

--- Top 20 najzastupljenijih pojmova u dataset-u ---
fraza/pojam  broj_pojavljivanja
        nan                 592
     године                 543
     godine                 516
        što                 321
        што                 300
        био                 249
        bio                 206
       била                 151
       kada                 147
  направите                 145
     takođe                 142
       када                 139
       bila                 138
  napravite                 135
     између                 135
      током                 134
        000                 129
       može                 129
     такође                 126
       било                 121


## Reading the recorded results

The frequency output includes `nan`. This presentation edit does not clean those values or recompute any statistics; the dataset remains the one used in the recorded experiment.